In [1]:
import pandas as pd
import numpy as np

# Load the integrated dataset created in Phase 4.
# This contains destination, places, weather,
# accommodation, and flight features.

data_path = "../data/cleaned/integrated_travel_dataset.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 50
Columns: 42


In [2]:
# Display the first few records to understand
# the current integrated dataset.

display(df.head())

,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,...,feels_like,humidity,pressure,wind_speed,cloudiness,weather_condition,weather_description,visibility,rain_1h,timestamp
0,Agra,13,60,98,42,32,3,1,0,0,...,34.57,94,1000,2.06,100,Clouds,overcast clouds,10000.0,0.00,2026-08-27T12:06:31+00:00
1,Ahmedabad,12,52,40,14,2,0,2,0,0,...,32.48,58,1004,3.09,100,Clouds,overcast clouds,10000.0,0.00,2026-08-27T12:10:33+00:00
2,Alappuzha,17,11,70,74,2,23,9,0,1,...,29.93,82,1012,4.60,100,Rain,light rain,10000.0,0.20,2026-08-27T12:11:40+00:00
3,Amritsar,10,14,56,41,9,1,0,0,0,...,41.97,59,998,1.54,16,Clouds,few clouds,10000.0,0.00,2026-08-27T12:11:38+00:00
4,Andaman,0,0,0,0,2,0,0,0,0,...,30.00,87,1009,7.98,100,Rain,light rain,10000.0,0.47,2026-08-27T12:22:29+00:00


In [3]:
# Check the data types of all columns.

print(df.dtypes)

destination                    str
sight_count                  int64
park_count                   int64
restaurant_count             int64
water_count                  int64
forest_count                 int64
wetland_count                int64
river_count                  int64
mountain_count               int64
coastal_count                int64
sand_count                   int64
protected_area_count         int64
hotel_count                float64
room_count                 float64
min_hotel_price            float64
avg_hotel_price            float64
max_hotel_price            float64
avg_allotment              float64
flight_count               float64
min_flight_price           float64
avg_flight_price           float64
max_flight_price           float64
avg_total_duration         float64
avg_outbound_stops         float64
avg_return_stops           float64
weather_available            int64
accommodation_available      int64
flight_available             int64
country             

In [4]:
# Check the amount of missing data in each column.

missing_values = df.isna().sum()

display(
    missing_values[missing_values > 0]
)

hotel_count           11
room_count            11
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
avg_allotment         11
flight_count          42
min_flight_price      42
avg_flight_price      42
max_flight_price      42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
dtype: int64

In [5]:
# Show destinations where accommodation data is unavailable.
# The availability flag tells us that these are genuine missing API results.

missing_accommodation = df[
    df["accommodation_available"] == 0
]

print(
    "Destinations without accommodation data:",
    len(missing_accommodation)
)

display(
    missing_accommodation[
        [
            "destination",
            "accommodation_available",
            "hotel_count",
            "avg_hotel_price"
        ]
    ]
)

Destinations without accommodation data: 11


,destination,accommodation_available,hotel_count,avg_hotel_price
4,Andaman,0,NaN,NaN
9,Coorg,0,NaN,NaN
21,Jim Corbett,0,NaN,NaN
23,Kaziranga,0,NaN,NaN
25,Kodaikanal,0,NaN,NaN
27,Ladakh,0,NaN,NaN
28,Mahabalipuram,0,NaN,NaN
36,Pahalgam,0,NaN,NaN
39,Ranchi,0,NaN,NaN
42,Shillong,0,NaN,NaN


In [6]:
# Show destinations where flight data is unavailable.
# These NaN values are expected because only 8 destinations
# were collected before reaching the API limit.

missing_flights = df[
    df["flight_available"] == 0
]

print(
    "Destinations without flight data:",
    len(missing_flights)
)

display(
    missing_flights[
        [
            "destination",
            "flight_available",
            "flight_count",
            "avg_flight_price"
        ]
    ]
)

Destinations without flight data: 42


,destination,flight_available,flight_count,avg_flight_price
0,Agra,0,NaN,NaN
1,Ahmedabad,0,NaN,NaN
2,Alappuzha,0,NaN,NaN
3,Amritsar,0,NaN,NaN
4,Andaman,0,NaN,NaN
6,Bhopal,0,NaN,NaN
7,Bhubaneswar,0,NaN,NaN
8,Chennai,0,NaN,NaN
9,Coorg,0,NaN,NaN
10,Darjeeling,0,NaN,NaN


In [7]:
# Select the main numerical features from accommodation and flight data.
# We inspect them before deciding on imputation and scaling.

price_duration_columns = [
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",
    "avg_allotment",
    "min_flight_price",
    "avg_flight_price",
    "max_flight_price",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops"
]

# Show descriptive statistics for these features.
display(
    df[price_duration_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
min_hotel_price,39.0,37.900256,35.218613,5.3300,18.015000,22.060000,46.780000,162.250000
avg_hotel_price,39.0,1973.969482,6377.518669,34.2963,68.430341,109.677500,181.071276,31449.736897
max_hotel_price,39.0,31475.033333,129964.269606,54.1600,164.840000,274.770000,1481.905000,802629.320000
avg_allotment,39.0,14.638156,10.223160,1.8000,7.770833,12.222222,17.169656,46.000000
min_flight_price,8.0,14302.500000,4138.848183,8776.0000,11427.250000,14458.500000,16542.000000,20498.000000
avg_flight_price,8.0,14302.500000,4138.848183,8776.0000,11427.250000,14458.500000,16542.000000,20498.000000
max_flight_price,8.0,14302.500000,4138.848183,8776.0000,11427.250000,14458.500000,16542.000000,20498.000000
avg_total_duration,8.0,358.342497,220.543061,148.7500,182.596154,269.913793,527.916667,705.000000
avg_outbound_stops,8.0,0.375000,0.517549,0.0000,0.000000,0.000000,1.000000,1.000000
avg_return_stops,8.0,0.295019,0.392032,0.0000,0.000000,0.068966,0.583333,1.000000


In [8]:
# Check accommodation price percentiles.
# Percentiles help us understand whether a few extreme values
# are strongly affecting the mean.

hotel_price_columns = [
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price"
]

hotel_percentiles = df[hotel_price_columns].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

display(hotel_percentiles)

,min_hotel_price,avg_hotel_price,max_hotel_price
0.25,18.0150,68.430341,164.840
0.50,22.0600,109.677500,274.770
0.75,46.7800,181.071276,1481.905
0.90,74.2000,3703.648886,94287.820
0.95,93.1240,8142.618083,94490.590
0.99,158.9098,28967.302546,534230.076


In [9]:
# Display destinations with high average hotel prices.
# This helps us determine whether extreme values are genuine
# expensive hotels or possible data-quality problems.

high_hotel_prices = df[
    df["avg_hotel_price"] >
    df["avg_hotel_price"].quantile(0.95)
][
    [
        "destination",
        "hotel_count",
        "min_hotel_price",
        "avg_hotel_price",
        "max_hotel_price"
    ]
].sort_values(
    "avg_hotel_price",
    ascending=False
)

display(high_hotel_prices)

,destination,hotel_count,min_hotel_price,avg_hotel_price,max_hotel_price
35,Ooty,4.0,76.44,31449.736897,94287.82
41,Rishikesh,6.0,20.96,24917.014921,802629.32


In [10]:
# Sort destinations by average hotel price.
# This helps us see how concentrated the extreme values are.

hotel_price_check = df[
    [
        "destination",
        "hotel_count",
        "room_count",
        "min_hotel_price",
        "avg_hotel_price",
        "max_hotel_price",
        "avg_allotment"
    ]
].sort_values(
    "avg_hotel_price",
    ascending=False
)

display(hotel_price_check)

,destination,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
35,Ooty,4.0,10.0,76.44,31449.736897,94287.82,2.482759
41,Rishikesh,6.0,14.0,20.96,24917.014921,802629.32,8.349206
3,Amritsar,6.0,13.0,21.21,6278.796212,94287.82,5.303030
12,Dharamshala,2.0,5.0,153.46,5912.065000,96315.52,3.352941
22,Jodhpur,4.0,17.0,16.40,3151.544857,9788.15,6.571429
26,Kolkata,14.0,45.0,13.88,1656.796456,94287.82,13.172996
14,Goa,11.0,41.0,19.13,359.648715,9926.91,14.730924
30,Mumbai,24.0,81.0,8.13,197.918615,7830.53,18.382892
2,Alappuzha,2.0,4.0,73.55,187.735556,335.48,12.222222
40,Ranthambore,1.0,3.0,162.25,183.883333,194.70,6.000000


In [11]:
# Inspect the destinations with the highest average hotel prices.
# We will compare their minimum, average, and maximum prices.

display(
    hotel_price_check.head(10)
)

,destination,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
35,Ooty,4.0,10.0,76.44,31449.736897,94287.82,2.482759
41,Rishikesh,6.0,14.0,20.96,24917.014921,802629.32,8.349206
3,Amritsar,6.0,13.0,21.21,6278.796212,94287.82,5.303030
12,Dharamshala,2.0,5.0,153.46,5912.065000,96315.52,3.352941
22,Jodhpur,4.0,17.0,16.40,3151.544857,9788.15,6.571429
26,Kolkata,14.0,45.0,13.88,1656.796456,94287.82,13.172996
14,Goa,11.0,41.0,19.13,359.648715,9926.91,14.730924
30,Mumbai,24.0,81.0,8.13,197.918615,7830.53,18.382892
2,Alappuzha,2.0,4.0,73.55,187.735556,335.48,12.222222
40,Ranthambore,1.0,3.0,162.25,183.883333,194.70,6.000000


In [13]:
# Load the original accommodation dataset.
# This is the source data from which the accommodation
# destination-level features were created.

accommodation_path = "../data/cleaned/accommodation_data.csv"

accommodation_df = pd.read_csv(accommodation_path)

print("Accommodation data loaded.")
print("Rows:", accommodation_df.shape[0])
print("Columns:", accommodation_df.shape[1])

print("\nColumns:")
print(accommodation_df.columns.tolist())

Accommodation data loaded.
Rows: 4835
Columns: 30

Columns:
['destination', 'check_in', 'check_out', 'hotel_code', 'hotel_name', 'hotel_category', 'hotel_category_name', 'destination_code', 'destination_name', 'zone_code', 'zone_name', 'hotel_latitude', 'hotel_longitude', 'room_code', 'room_name', 'rate_key', 'rate_class', 'rate_type', 'price', 'allotment', 'payment_type', 'board_code', 'board_name', 'rooms', 'adults', 'children', 'offer_name', 'offer_amount', 'cancellation_amount', 'cancellation_from']


In [14]:
# Inspect the original accommodation records for destinations
# where unusually large hotel prices were found.
#
# We are checking the raw values before deciding whether
# they are valid, outliers, or data-quality problems.

outlier_destinations = ["Ooty", "Rishikesh"]

outlier_records = accommodation_df[
    accommodation_df["destination"].isin(outlier_destinations)
].copy()

print("Underlying records:", len(outlier_records))

display(
    outlier_records[
        [
            "destination",
            "hotel_name",
            "hotel_category",
            "hotel_category_name",
            "room_name",
            "price",
            "allotment",
            "rooms",
            "adults",
            "children"
        ]
    ].sort_values(
        ["destination", "price"],
        ascending=[True, False]
    ).head(50)
)

Underlying records: 155


,destination,hotel_name,hotel_category,hotel_category_name,room_name,price,allotment,rooms,adults,children
1679,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class C villa Orchid SE,94287.82,1,1,1,0
1681,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Chalet,94287.82,1,1,1,0
1683,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class A villa Chalet,94287.82,1,1,1,0
1685,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class B villa British,94287.82,1,1,1,0
1687,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class F Villa Royal Swiss,94287.82,1,1,1,0
1678,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class C villa Orchid SE,87687.67,1,1,1,0
1680,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Chalet,87687.67,1,1,1,0
1682,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class A villa Chalet,87687.67,1,1,1,0
1684,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class B villa British,87687.67,1,1,1,0
1686,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class F Villa Royal Swiss,87687.67,1,1,1,0


In [16]:
# Load environment variables from the project's .env file.
# This keeps the API credentials outside the notebook code.

import os
from dotenv import load_dotenv

load_dotenv("../.env")

HOTELBEDS_API_KEY = os.getenv("HOTELBEDS_API_KEY")
HOTELBEDS_SECRET = os.getenv("HOTELBEDS_SECRET")

print("API key loaded:", HOTELBEDS_API_KEY is not None)
print("Secret loaded:", HOTELBEDS_SECRET is not None)

API key loaded: True
Secret loaded: True


In [19]:
import hashlib
import time

# Create the current Unix timestamp.
# Hotelbeds uses the current timestamp as part of the X-Signature.
timestamp = str(int(time.time()))

# Combine the API key, secret, and timestamp.
signature_string = HOTELBEDS_API_KEY + HOTELBEDS_SECRET + timestamp

# Generate the SHA-256 hash required by Hotelbeds.
hotelbeds_signature = hashlib.sha256(
    signature_string.encode("utf-8")
).hexdigest()

# Show only safe information for verification.
# We never print the API key, secret, or complete signature.
print("Hotelbeds timestamp generated:", timestamp)
print("Hotelbeds signature generated:", bool(hotelbeds_signature))

Hotelbeds timestamp generated: 1787837360
Hotelbeds signature generated: True


In [22]:
# Create the headers required by Hotelbeds.

headers = {
    "Api-key": HOTELBEDS_API_KEY,
    "X-Signature": hotelbeds_signature,
    "Accept": "application/json",
    "Content-Type": "application/json"
}

print("Headers created successfully.")

Headers created successfully.


In [ ]:
import requests

# Hotelbeds test API endpoint.
url = "https://api.test.hotelbeds.com/hotel-api/1.0/hotels"

# One-night test booking request.
payload = {
    "stay": {
        "checkIn": "2026-09-15",
        "checkOut": "2026-09-16"
    },
    "occupancies": [
        {
            "rooms": 1,
            "adults": 1,
            "children": 0
        }
    ],
    "geolocation": {
        "latitude": 11.4102,
        "longitude": 76.6950,
        "radius": 20,
        "unit": "km"
    }
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=30
)

print("Status code:", response.status_code)

response.raise_for_status()

data = response.json()

print("API request successful.")

In [24]:
# Inspect the first returned hotel/rate.
# We are looking specifically for currency information.

hotels = data.get("hotels", {}).get("hotels", [])

if not hotels:
    print("No hotels returned.")
else:
    hotel = hotels[0]

    print("Hotel:", hotel.get("name"))

    rooms = hotel.get("rooms", [])

    if not rooms:
        print("No rooms returned.")
    else:
        rates = rooms[0].get("rates", [])

        if not rates:
            print("No rates returned.")
        else:
            rate = rates[0]

            print("\nFirst rate:")
            print(rate)

Hotel: Neemrana's Wallwood Garden

First rate:
{'rateKey': '20260915|20260916|W|270|766917|DBL.DX|BAR-CP|BB||1~1~0||P@07~~23858~1024211153~N~~~NOR~~C0CC8FF10847475178783746066205AAUK0004000000001022c4c', 'rateClass': 'NOR', 'rateType': 'BOOKABLE', 'net': '76.44', 'allotment': 3, 'paymentType': 'AT_WEB', 'packaging': True, 'boardCode': 'BB', 'boardName': 'BED AND BREAKFAST', 'cancellationPolicies': [{'amount': '76.44', 'from': '2026-09-11T23:59:00+05:30'}], 'rooms': 1, 'adults': 1, 'children': 0, 'offers': [{'code': '9005', 'name': 'Exclusive discount', 'amount': '-11.43'}]}


In [25]:
# Inspect the top-level response structure.
# We are looking for currency or pricing information
# outside the individual rate object.

print("Top-level keys:")
print(data.keys())

print("\nFull top-level information:")
for key, value in data.items():
    if key != "hotels":
        print(f"{key}: {value}")

Top-level keys:
dict_keys(['auditData', 'hotels'])

Full top-level information:
auditData: {'processTime': '126', 'timestamp': '2026-08-27 13:31:00.788', 'requestHost': '', 'serverId': '', 'environment': '', 'release': '', 'token': 'EC6CA66763E542728FACA7E77A3BB998', 'internal': '0|C0CC8FF10847475178783746066205|UK|10|4|29||||||||||||36||1~1~1~0|0|0||0|6d946d3a994e9c13976dbe2b8832a837||||'}


In [26]:
# Check the first hotel's complete basic information.
# This may reveal information related to the destination,
# currency, or pricing configuration.

print(hotels[0].keys())

dict_keys(['code', 'name', 'categoryCode', 'categoryName', 'destinationCode', 'destinationName', 'zoneCode', 'zoneName', 'latitude', 'longitude', 'rooms', 'minRate', 'maxRate', 'currency'])


In [27]:
# Check whether the response contains any currency-related
# information anywhere in the first hotel's structure.

import json

hotel_text = json.dumps(hotels[0], indent=2)

for line in hotel_text.splitlines():
    if "curr" in line.lower():
        print(line)

  "currency": "EUR"


In [28]:
# Display every raw accommodation price.
# The source price is in EUR, as confirmed from the Hotelbeds API response.

price_check = accommodation_df[
    [
        "destination",
        "hotel_name",
        "room_name",
        "price",
        "allotment",
        "rooms",
        "adults",
        "children"
    ]
].sort_values(
    "price",
    ascending=False
)

display(price_check)

,destination,hotel_name,room_name,price,allotment,rooms,adults,children
1195,Rishikesh,Anantham Rishikesh,DOUBLE DELUXE,802629.32,1,1,1,0
1197,Rishikesh,Anantham Rishikesh,DOUBLE COMFORT,802629.32,1,1,1,0
1194,Rishikesh,Anantham Rishikesh,DOUBLE DELUXE,746445.27,1,1,1,0
1196,Rishikesh,Anantham Rishikesh,DOUBLE COMFORT,746445.27,1,1,1,0
3144,Dharamshala,Hill Ventures,101,96315.52,1,1,1,0
...,...,...,...,...,...,...,...,...
1333,Nainital,Goroomgo Moon Nainital,Deluxe Double Bed Room,8.94,30,1,1,0
2349,Mumbai,Nap Manor Hostels,One bed in a 8-BED Female Dorm with sharing wa...,8.13,8,1,1,0
1050,Agra,MTJ Hotels Agra,Bed in 6 Bed mixed Dormitory Room,7.85,12,1,1,0
1049,Agra,MTJ Hotels Agra,Bed in 6 Bed mixed Dormitory Room,7.30,12,1,1,0


In [29]:
# Show the 50 lowest accommodation prices.

display(
    price_check.sort_values("price").head(50)
)

,destination,hotel_name,room_name,price,allotment,rooms,adults,children
1048,Agra,MTJ Hotels Agra,Bed in 6 Bed mixed Dormitory Room,5.33,12,1,1,0
1049,Agra,MTJ Hotels Agra,Bed in 6 Bed mixed Dormitory Room,7.30,12,1,1,0
1050,Agra,MTJ Hotels Agra,Bed in 6 Bed mixed Dormitory Room,7.85,12,1,1,0
2349,Mumbai,Nap Manor Hostels,One bed in a 8-BED Female Dorm with sharing wa...,8.13,8,1,1,0
1333,Nainital,Goroomgo Moon Nainital,Deluxe Double Bed Room,8.94,30,1,1,0
2350,Mumbai,Nap Manor Hostels,One bed in a 8-BED Female Dorm with sharing wa...,9.03,8,1,1,0
2355,Mumbai,Nap Manor Hostels,One bed in 8-Bed Mixed Dorm Ground Floor,9.30,15,1,1,0
2353,Mumbai,Nap Manor Hostels,One bed in a 8-Bed Mixed Dorm First Floor,9.30,7,1,1,0
2351,Mumbai,Nap Manor Hostels,One bed in a 6-BED Mixed Dorm Sharing Washroom,9.30,6,1,1,0
2357,Mumbai,Nap Manor Hostels,One bed in a 4-bed Mixed Dorm (Mini),9.30,3,1,1,0


In [37]:
# Show the 50 highest accommodation prices.

display(
    price_check.sort_values(
        "price",
        ascending=False
    ).head(60)
)

,destination,hotel_name,room_name,price,allotment,rooms,adults,children
1195,Rishikesh,Anantham Rishikesh,DOUBLE DELUXE,802629.32,1,1,1,0
1197,Rishikesh,Anantham Rishikesh,DOUBLE COMFORT,802629.32,1,1,1,0
1194,Rishikesh,Anantham Rishikesh,DOUBLE DELUXE,746445.27,1,1,1,0
1196,Rishikesh,Anantham Rishikesh,DOUBLE COMFORT,746445.27,1,1,1,0
3144,Dharamshala,Hill Ventures,101,96315.52,1,1,1,0
3145,Dharamshala,Hill Ventures,101,96315.52,1,1,1,0
3108,Amritsar,Hotel Jk Residency (7 Min Walk From Golden Tem...,TRIPLE DELUXE ROOM,94287.82,1,1,1,0
3106,Amritsar,Hotel Jk Residency (7 Min Walk From Golden Tem...,DELUXE ROOM,94287.82,1,1,1,0
1679,Ooty,The Frame Resorts Ooty,Class C villa Orchid SE,94287.82,1,1,1,0
3292,Kolkata,Dover Inn By Bookmerihotel.com,Deluxe Double,94287.82,1,1,1,0


In [38]:
# Divide the accommodation prices into useful ranges.
# This helps us understand how many records fall into
# normal-looking versus extremely high price ranges.

price_ranges = pd.cut(
    accommodation_df["price"],
    bins=[
        0,
        50,
        100,
        200,
        500,
        1000,
        5000,
        10000,
        50000,
        100000,
        500000,
        float("inf")
    ]
)

print(
    price_ranges.value_counts().sort_index()
)

price
(0.0, 50.0]             1878
(50.0, 100.0]           1668
(100.0, 200.0]           811
(200.0, 500.0]           230
(500.0, 1000.0]          186
(1000.0, 5000.0]           4
(5000.0, 10000.0]         34
(10000.0, 50000.0]         0
(50000.0, 100000.0]       20
(100000.0, 500000.0]       0
(500000.0, inf]            4
Name: count, dtype: int64


In [39]:
# Inspect all accommodation records with prices above €5,000.
# We want to understand exactly what these unusual records represent
# before converting any prices to INR.

high_price_records = accommodation_df[
    accommodation_df["price"] > 5000
].copy()

print("Records above €5,000:", len(high_price_records))

display(
    high_price_records[
        [
            "destination",
            "hotel_name",
            "hotel_category",
            "hotel_category_name",
            "room_name",
            "price",
            "allotment",
            "rooms",
            "adults",
            "children",
            "board_name",
            "payment_type"
        ]
    ].sort_values(
        "price",
        ascending=False
    )
)

Records above €5,000: 58


,destination,hotel_name,hotel_category,hotel_category_name,room_name,price,allotment,rooms,adults,children,board_name,payment_type
1195,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE DELUXE,802629.32,1,1,1,0,ROOM ONLY,AT_WEB
1197,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE COMFORT,802629.32,1,1,1,0,ROOM ONLY,AT_WEB
1196,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE COMFORT,746445.27,1,1,1,0,ROOM ONLY,AT_WEB
1194,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE DELUXE,746445.27,1,1,1,0,ROOM ONLY,AT_WEB
3144,Dharamshala,Hill Ventures,4EST,4 STARS,101,96315.52,1,1,1,0,ROOM ONLY,AT_WEB
3145,Dharamshala,Hill Ventures,4EST,4 STARS,101,96315.52,1,1,1,0,ROOM ONLY,AT_WEB
1683,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class A villa Chalet,94287.82,1,1,1,0,BED AND BREAKFAST,AT_WEB
1681,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Chalet,94287.82,1,1,1,0,BED AND BREAKFAST,AT_WEB
1679,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class C villa Orchid SE,94287.82,1,1,1,0,BED AND BREAKFAST,AT_WEB
3108,Amritsar,Hotel Jk Residency (7 Min Walk From Golden Tem...,3EST,3 STARS,TRIPLE DELUXE ROOM,94287.82,1,1,1,0,ROOM ONLY,AT_WEB


In [40]:
# Count unusually high-price records by destination.

high_price_by_destination = (
    high_price_records
    .groupby("destination")
    .agg(
        high_price_records=("price", "count"),
        min_high_price=("price", "min"),
        max_high_price=("price", "max"),
        avg_high_price=("price", "mean")
    )
    .sort_values(
        "high_price_records",
        ascending=False
    )
)

display(high_price_by_destination)

,high_price_records,min_high_price,max_high_price,avg_high_price
destination,,,,
Jodhpur,11,9788.15,9788.15,9788.1500
Amritsar,10,8009.07,94287.82,41200.5400
Ooty,10,87687.67,94287.82,90987.7450
Goa,8,7012.54,9926.91,9295.7125
Delhi,5,6655.95,7642.72,6853.3040
Mumbai,4,7830.53,7830.53,7830.5300
Kolkata,4,87687.67,94287.82,90987.7450
Rishikesh,4,746445.27,802629.32,774537.2950
Dharamshala,2,96315.52,96315.52,96315.5200


,destination,hotel_name,hotel_category,hotel_category_name,room_name,price,allotment,rooms,adults,children,board_name,payment_type
1195,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE DELUXE,802629.32,1,1,1,0,ROOM ONLY,AT_WEB
1197,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE COMFORT,802629.32,1,1,1,0,ROOM ONLY,AT_WEB
1196,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE COMFORT,746445.27,1,1,1,0,ROOM ONLY,AT_WEB
1194,Rishikesh,Anantham Rishikesh,2EST,2 STARS,DOUBLE DELUXE,746445.27,1,1,1,0,ROOM ONLY,AT_WEB
3144,Dharamshala,Hill Ventures,4EST,4 STARS,101,96315.52,1,1,1,0,ROOM ONLY,AT_WEB
3145,Dharamshala,Hill Ventures,4EST,4 STARS,101,96315.52,1,1,1,0,ROOM ONLY,AT_WEB
1683,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class A villa Chalet,94287.82,1,1,1,0,BED AND BREAKFAST,AT_WEB
1681,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Chalet,94287.82,1,1,1,0,BED AND BREAKFAST,AT_WEB
1679,Ooty,The Frame Resorts Ooty,4EST,4 STARS,Class C villa Orchid SE,94287.82,1,1,1,0,BED AND BREAKFAST,AT_WEB
3108,Amritsar,Hotel Jk Residency (7 Min Walk From Golden Tem...,3EST,3 STARS,TRIPLE DELUXE ROOM,94287.82,1,1,1,0,ROOM ONLY,AT_WEB


Normal records (price <= €5,000):
4777

High-price records (price > €5,000):
58


In [44]:
# Examine characteristics of the extreme-price records.
# We are looking for a systematic pattern in the API data.

high_price_records[
    [
        "rate_class",
        "rate_type",
        "payment_type",
        "board_name",
        "hotel_category_name",
        "price"
    ]
].sort_values(
    "price",
    ascending=False
)

,rate_class,rate_type,payment_type,board_name,hotel_category_name,price
1195,NOR,BOOKABLE,AT_WEB,ROOM ONLY,2 STARS,802629.32
1197,NOR,BOOKABLE,AT_WEB,ROOM ONLY,2 STARS,802629.32
1196,NOR,BOOKABLE,AT_WEB,ROOM ONLY,2 STARS,746445.27
1194,NOR,BOOKABLE,AT_WEB,ROOM ONLY,2 STARS,746445.27
3144,NOR,BOOKABLE,AT_WEB,ROOM ONLY,4 STARS,96315.52
3145,NOR,BOOKABLE,AT_WEB,ROOM ONLY,4 STARS,96315.52
1683,NOR,BOOKABLE,AT_WEB,BED AND BREAKFAST,4 STARS,94287.82
1681,NOR,BOOKABLE,AT_WEB,BED AND BREAKFAST,4 STARS,94287.82
1679,NOR,BOOKABLE,AT_WEB,BED AND BREAKFAST,4 STARS,94287.82
3108,NOR,BOOKABLE,AT_WEB,ROOM ONLY,3 STARS,94287.82


In [46]:
# ---------------------------------------------------------
# Standardize accommodation prices to INR
# ---------------------------------------------------------
#
# Project rule:
#   - Prices <= 5000 are treated as EUR and converted to INR.
#   - Prices > 5000 are treated as already being INR.
#
# We keep the original `price` column unchanged so that
# the raw collected data remains traceable.
# ---------------------------------------------------------

EUR_TO_INR = 100.0   # Project conversion rate

accommodation_df["price_inr"] = accommodation_df["price"]

# Identify prices that we are treating as EUR.
eur_mask = accommodation_df["price"] <= 5000

# Convert only those prices from EUR to INR.
accommodation_df.loc[eur_mask, "price_inr"] = (
    accommodation_df.loc[eur_mask, "price"] * EUR_TO_INR
)

print("Total accommodation records:", len(accommodation_df))
print("Prices treated as EUR:", eur_mask.sum())
print("Prices treated as INR:", (~eur_mask).sum())

Total accommodation records: 4835
Prices treated as EUR: 4777
Prices treated as INR: 58


In [48]:
# Compare original and standardized prices.

price_comparison = accommodation_df[
    ["destination", "hotel_name", "room_name", "price", "price_inr"]
].sort_values(
    "price_inr",
    ascending=False
)

display(price_comparison.head(30))

,destination,hotel_name,room_name,price,price_inr
1195,Rishikesh,Anantham Rishikesh,DOUBLE DELUXE,802629.32,802629.32
1197,Rishikesh,Anantham Rishikesh,DOUBLE COMFORT,802629.32,802629.32
1196,Rishikesh,Anantham Rishikesh,DOUBLE COMFORT,746445.27,746445.27
1194,Rishikesh,Anantham Rishikesh,DOUBLE DELUXE,746445.27,746445.27
726,Jaipur,Rajmahal Palace RAAS Jaipur,The Maharani Suite,2000.66,200066.00
724,Jaipur,Rajmahal Palace RAAS Jaipur,The Maharaja Suite,1650.54,165054.00
725,Jaipur,Rajmahal Palace RAAS Jaipur,The Maharani Suite,1598.17,159817.00
723,Jaipur,Rajmahal Palace RAAS Jaipur,The Maharaja Suite,1598.17,159817.00
3145,Dharamshala,Hill Ventures,101,96315.52,96315.52
3144,Dharamshala,Hill Ventures,101,96315.52,96315.52


In [49]:
# ---------------------------------------------------------
# Recalculate destination-level accommodation features
# using the standardized INR price.
# ---------------------------------------------------------

accommodation_features = (
    accommodation_df
    .groupby("destination")
    .agg(
        hotel_count=("hotel_code", "nunique"),
        room_count=("room_code", "nunique"),

        # Accommodation prices are now standardized to INR.
        min_hotel_price=("price_inr", "min"),
        avg_hotel_price=("price_inr", "mean"),
        max_hotel_price=("price_inr", "max"),

        # Average number of rooms available for booking.
        avg_allotment=("allotment", "mean")
    )
    .reset_index()
)

print("Accommodation feature table created.")
print("Rows:", len(accommodation_features))
print("Columns:", len(accommodation_features.columns))

display(accommodation_features.head())

Accommodation feature table created.
Rows: 39
Columns: 7


,destination,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
0,Agra,4,11,533.0,3586.227273,6095.00,16.000000
1,Ahmedabad,7,35,2653.0,10785.250000,27477.00,16.558824
2,Alappuzha,2,4,7355.0,18773.555556,33548.00,12.222222
3,Amritsar,6,13,2121.0,9871.521212,94287.82,5.303030
4,Bengaluru,18,68,1589.0,10621.343137,78304.00,19.799020


In [50]:
# ---------------------------------------------------------
# Check the recalculated accommodation prices.
# ---------------------------------------------------------

display(
    accommodation_features[
        [
            "destination",
            "hotel_count",
            "room_count",
            "min_hotel_price",
            "avg_hotel_price",
            "max_hotel_price",
            "avg_allotment"
        ]
    ]
    .sort_values("avg_hotel_price", ascending=False)
)

,destination,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
32,Rishikesh,6,14,2096.0,57441.422063,802629.32,8.349206
28,Ooty,4,10,7644.0,38840.325862,94287.82,2.482759
10,Dharamshala,2,5,15346.0,30310.236471,96315.52,3.352941
2,Alappuzha,2,4,7355.0,18773.555556,33548.00,12.222222
31,Ranthambore,1,3,16225.0,18388.333333,19470.00,6.000000
37,Varanasi,3,13,3979.0,17825.921875,45098.00,12.062500
27,Nainital,6,25,894.0,16253.942529,96315.00,8.540230
24,Munnar,2,4,7364.0,14630.200000,27801.00,9.200000
20,Kochi,7,21,3128.0,14246.865169,64020.00,12.078652
21,Kolkata,14,45,1388.0,13649.489367,94287.82,13.172996


In [52]:
# Load the integrated 50-destination dataset.
integrated_df = pd.read_csv(
    "../data/cleaned/integrated_travel_dataset.csv"
)

print("Integrated dataset loaded.")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

Integrated dataset loaded.
Rows: 50
Columns: 42


In [53]:
# ---------------------------------------------------------
# Replace the old accommodation features with the newly
# recalculated INR-based accommodation features.
# ---------------------------------------------------------

# These are the accommodation columns currently present
# in the integrated dataset.
accommodation_columns = [
    "hotel_count",
    "room_count",
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",
    "avg_allotment"
]

# Remove the old accommodation values.
integrated_df = integrated_df.drop(
    columns=accommodation_columns,
    errors="ignore"
)

# Merge the newly calculated accommodation features.
integrated_df = integrated_df.merge(
    accommodation_features,
    on="destination",
    how="left"
)

print("Accommodation features updated successfully.")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

Accommodation features updated successfully.
Rows: 50
Columns: 42


In [54]:
# ---------------------------------------------------------
# Verify the newly merged accommodation features.
# ---------------------------------------------------------

print("Accommodation availability:")
print(
    integrated_df["accommodation_available"]
    .value_counts()
    .sort_index()
)

print("\nMissing accommodation values:")
print(
    integrated_df[
        [
            "hotel_count",
            "room_count",
            "min_hotel_price",
            "avg_hotel_price",
            "max_hotel_price",
            "avg_allotment"
        ]
    ].isna().sum()
)

print("\nDuplicate destinations:")
print(
    integrated_df["destination"].duplicated().sum()
)

Accommodation availability:
accommodation_available
0    11
1    39
Name: count, dtype: int64

Missing accommodation values:
hotel_count        11
room_count         11
min_hotel_price    11
avg_hotel_price    11
max_hotel_price    11
avg_allotment      11
dtype: int64

Duplicate destinations:
0


In [55]:
# ---------------------------------------------------------
# Display the final accommodation features after
# currency standardization.
# ---------------------------------------------------------

display(
    integrated_df[
        [
            "destination",
            "accommodation_available",
            "hotel_count",
            "room_count",
            "min_hotel_price",
            "avg_hotel_price",
            "max_hotel_price",
            "avg_allotment"
        ]
    ].sort_values("avg_hotel_price")
)

,destination,accommodation_available,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
18,Indore,1,3.0,19.0,2206.0,3429.630000,5657.00,16.520000
0,Agra,1,4.0,11.0,533.0,3586.227273,6095.00,16.000000
6,Bhopal,1,2.0,9.0,2732.0,3924.625000,5416.00,7.541667
45,Thiruvananthapuram,1,2.0,14.0,1918.0,4164.044118,19184.00,15.911765
37,Pondicherry,1,4.0,14.0,1787.0,5185.289474,21556.00,29.421053
10,Darjeeling,1,7.0,13.0,1882.0,5240.978495,11287.00,5.688172
46,Udaipur,1,10.0,22.0,1382.0,5742.202614,24581.00,12.653595
19,Jaipur,1,24.0,71.0,1724.0,6318.798780,200066.00,12.404472
14,Goa,1,11.0,41.0,1913.0,6397.785944,24689.00,14.730924
15,Gokarna,1,1.0,5.0,6181.0,6563.500000,6905.00,1.800000


In [56]:
# ---------------------------------------------------------
# Save the current preprocessing checkpoint.
#
# We use a NEW filename so the original integrated dataset
# remains untouched and can always be recovered.
# ---------------------------------------------------------

output_path = "../data/cleaned/travel_integrated_preprocessed.csv"

integrated_df.to_csv(
    output_path,
    index=False
)

print("Preprocessed dataset saved successfully.")
print("Path:", output_path)
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

Preprocessed dataset saved successfully.
Path: ../data/cleaned/travel_integrated_preprocessed.csv
Rows: 50
Columns: 42


In [93]:
# ---------------------------------------------------------
# Load the preprocessing checkpoint we just created.
# ---------------------------------------------------------

import pandas as pd

processed_path = "../data/cleaned/travel_integrated_preprocessed.csv"

integrated_df = pd.read_csv(processed_path)

print("Dataset loaded.")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

Dataset loaded.
Rows: 50
Columns: 42


In [94]:
# ---------------------------------------------------------
# Check how many destinations have flight data.
# ---------------------------------------------------------

print("Flight availability:")
print(
    integrated_df["flight_available"]
    .value_counts()
    .sort_index()
)

Flight availability:
flight_available
0    42
1     8
Name: count, dtype: int64


In [95]:
# ---------------------------------------------------------
# Display the destinations for which we actually collected
# flight information.
# ---------------------------------------------------------

flight_columns = [
    "destination",
    "flight_available",
    "flight_count",
    "min_flight_price",
    "avg_flight_price",
    "max_flight_price",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops"
]

display(
    integrated_df[
        integrated_df["flight_available"] == 1
    ][flight_columns]
)

,destination,flight_available,flight_count,min_flight_price,avg_flight_price,max_flight_price,avg_total_duration,avg_outbound_stops,avg_return_stops
5,Bengaluru,1,13.0,8776.0,8776.0,8776.0,160.384615,0.0,0.000000
11,Delhi,1,29.0,14785.0,14785.0,14785.0,299.827586,0.0,0.137931
14,Goa,1,4.0,9538.0,9538.0,9538.0,148.750000,0.0,0.000000
19,Jaipur,1,3.0,15767.0,15767.0,15767.0,240.000000,0.0,0.000000
30,Mumbai,1,17.0,12057.0,12057.0,12057.0,190.000000,0.0,0.000000
31,Munnar,1,18.0,14132.0,14132.0,14132.0,494.444444,1.0,0.555556
44,Srinagar,1,3.0,18867.0,18867.0,18867.0,705.000000,1.0,1.000000
46,Udaipur,1,3.0,20498.0,20498.0,20498.0,628.333333,1.0,0.666667


In [96]:
# ---------------------------------------------------------
# Verify the missing flight records.
#
# Missing flight information should remain NaN.
# We must NOT replace missing flight prices with 0,
# because ₹0 would incorrectly mean "free flight".
# ---------------------------------------------------------

missing_flights = integrated_df[
    integrated_df["flight_available"] == 0
]

display(
    missing_flights[
        [
            "destination",
            "flight_available",
            "flight_count",
            "min_flight_price",
            "avg_flight_price",
            "max_flight_price",
            "avg_total_duration",
            "avg_outbound_stops",
            "avg_return_stops"
        ]
    ]
)

,destination,flight_available,flight_count,min_flight_price,avg_flight_price,max_flight_price,avg_total_duration,avg_outbound_stops,avg_return_stops
0,Agra,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ahmedabad,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Alappuzha,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amritsar,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andaman,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Bhopal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Bhubaneswar,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chennai,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Coorg,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Darjeeling,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
# ---------------------------------------------------------
# Prepare flight features for machine learning.
#
# IMPORTANT:
# A value of 0 in these columns does NOT mean:
# "there are free flights" or "flight duration is zero".
#
# The separate `flight_available` flag tells the model
# whether the flight information actually exists.
# ---------------------------------------------------------

flight_columns = [
    "flight_count",
    "min_flight_price",
    "avg_flight_price",
    "max_flight_price",
    "avg_total_duration",
    "avg_outbound_stops",
    "avg_return_stops"
]

# Fill missing flight values with 0.
# flight_available preserves the information that these
# values were unavailable rather than genuinely zero.
integrated_df[flight_columns] = (
    integrated_df[flight_columns].fillna(0)
)

print("Flight missing values handled.")

print("\nRemaining missing flight values:")
print(
    integrated_df[flight_columns]
    .isna()
    .sum()
)

Flight missing values handled.

Remaining missing flight values:
flight_count          0
min_flight_price      0
avg_flight_price      0
max_flight_price      0
avg_total_duration    0
avg_outbound_stops    0
avg_return_stops      0
dtype: int64


In [98]:
# ---------------------------------------------------------
# Verify that flight availability information is still
# preserved after filling the numerical missing values.
# ---------------------------------------------------------

display(
    integrated_df[
        [
            "destination",
            "flight_available",
            "flight_count",
            "avg_flight_price",
            "avg_total_duration"
        ]
    ].head(15)
)

,destination,flight_available,flight_count,avg_flight_price,avg_total_duration
0,Agra,0,0.0,0.0,0.000000
1,Ahmedabad,0,0.0,0.0,0.000000
2,Alappuzha,0,0.0,0.0,0.000000
3,Amritsar,0,0.0,0.0,0.000000
4,Andaman,0,0.0,0.0,0.000000
5,Bengaluru,1,13.0,8776.0,160.384615
6,Bhopal,0,0.0,0.0,0.000000
7,Bhubaneswar,0,0.0,0.0,0.000000
8,Chennai,0,0.0,0.0,0.000000
9,Coorg,0,0.0,0.0,0.000000


In [99]:
# ---------------------------------------------------------
# Prepare accommodation features for machine learning.
#
# IMPORTANT:
# A zero here does NOT mean:
# "there are zero hotels" or "hotel price is zero".
#
# The separate `accommodation_available` flag tells the
# model that the actual accommodation information was
# unavailable for these destinations.
# ---------------------------------------------------------

accommodation_columns = [
    "hotel_count",
    "room_count",
    "min_hotel_price",
    "avg_hotel_price",
    "max_hotel_price",
    "avg_allotment"
]

# Fill missing accommodation values with 0.
integrated_df[accommodation_columns] = (
    integrated_df[accommodation_columns].fillna(0)
)

print("Accommodation missing values handled.")

print("\nRemaining missing accommodation values:")
print(
    integrated_df[accommodation_columns]
    .isna()
    .sum()
)

Accommodation missing values handled.

Remaining missing accommodation values:
hotel_count        0
room_count         0
min_hotel_price    0
avg_hotel_price    0
max_hotel_price    0
avg_allotment      0
dtype: int64


In [100]:
# ---------------------------------------------------------
# Verify that the accommodation availability flag still
# correctly identifies the destinations where data was
# unavailable.
# ---------------------------------------------------------

display(
    integrated_df[
        [
            "destination",
            "accommodation_available",
            "hotel_count",
            "avg_hotel_price",
            "max_hotel_price"
        ]
    ]
)

,destination,accommodation_available,hotel_count,avg_hotel_price,max_hotel_price
0,Agra,1,4.0,3586.227273,6095.00
1,Ahmedabad,1,7.0,10785.250000,27477.00
2,Alappuzha,1,2.0,18773.555556,33548.00
3,Amritsar,1,6.0,9871.521212,94287.82
4,Andaman,0,0.0,0.000000,0.00
5,Bengaluru,1,18.0,10621.343137,78304.00
6,Bhopal,1,2.0,3924.625000,5416.00
7,Bhubaneswar,1,3.0,7797.000000,24848.00
8,Chennai,1,9.0,6569.234899,18214.00
9,Coorg,0,0.0,0.000000,0.00


In [101]:
# ---------------------------------------------------------
# Check every column in the integrated dataset for missing
# values after handling flight and accommodation data.
# ---------------------------------------------------------

missing_summary = (
    integrated_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by column:")
display(
    missing_summary[missing_summary > 0]
)

Missing values by column:


Series([], dtype: int64)

In [102]:
# ---------------------------------------------------------
# Final structural validation
# ---------------------------------------------------------

print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

print(
    "Duplicate destinations:",
    integrated_df["destination"].duplicated().sum()
)

print("\nMissing values:")
print(integrated_df.isna().sum().sum())

print("\nData types:")
print(integrated_df.dtypes)

Rows: 50
Columns: 42
Duplicate destinations: 0

Missing values:
0

Data types:
destination                    str
sight_count                  int64
park_count                   int64
restaurant_count             int64
water_count                  int64
forest_count                 int64
wetland_count                int64
river_count                  int64
mountain_count               int64
coastal_count                int64
sand_count                   int64
protected_area_count         int64
flight_count               float64
min_flight_price           float64
avg_flight_price           float64
max_flight_price           float64
avg_total_duration         float64
avg_outbound_stops         float64
avg_return_stops           float64
weather_available            int64
accommodation_available      int64
flight_available             int64
country                        str
latitude                   float64
longitude                  float64
temperature                float64
feels_like 

In [103]:
# ---------------------------------------------------------
# Inspect categorical columns before encoding.
# ---------------------------------------------------------

categorical_columns = [
    "country",
    "weather_condition",
    "weather_description"
]

for column in categorical_columns:
    print("\n" + "=" * 50)
    print(f"{column}")
    print("=" * 50)
    print("Unique values:", integrated_df[column].nunique())
    print(integrated_df[column].value_counts())


country
Unique values: 1
country
IN    50
Name: count, dtype: int64

weather_condition
Unique values: 3
weather_condition
Clouds    31
Rain      16
Clear      3
Name: count, dtype: int64

weather_description
Unique values: 7
weather_description
overcast clouds     20
light rain          12
few clouds           4
moderate rain        4
broken clouds        4
scattered clouds     3
clear sky            3
Name: count, dtype: int64


In [104]:
# ---------------------------------------------------------
# Inspect timestamp format before deciding whether it
# should be removed or transformed.
# ---------------------------------------------------------

print("\nTimestamp examples:")
display(
    integrated_df[
        ["destination", "timestamp"]
    ].head(10)
)


Timestamp examples:


,destination,timestamp
0,Agra,2026-08-27T12:06:31+00:00
1,Ahmedabad,2026-08-27T12:10:33+00:00
2,Alappuzha,2026-08-27T12:11:40+00:00
3,Amritsar,2026-08-27T12:11:38+00:00
4,Andaman,2026-08-27T12:22:29+00:00
5,Bengaluru,2026-08-27T12:03:34+00:00
6,Bhopal,2026-08-27T12:11:08+00:00
7,Bhubaneswar,2026-08-27T12:05:18+00:00
8,Chennai,2026-08-27T11:57:01+00:00
9,Coorg,2026-08-27T12:20:36+00:00


In [105]:
# ---------------------------------------------------------
# Create a separate dataframe for ML preprocessing.
#
# We keep `integrated_df` unchanged because it is our
# complete integrated dataset and contains useful information
# such as the destination name and timestamp.
# ---------------------------------------------------------

ml_df = integrated_df.copy()

# `country` has only one unique value: IN.
# A constant feature provides no information to the model.
#
# `timestamp` represents when we collected the weather data.
# It is not a destination/travel preference feature.
#
# We therefore remove both from the ML feature dataframe.
ml_df = ml_df.drop(
    columns=[
        "country",
        "timestamp"
    ]
)

print("ML dataframe created.")
print("Rows:", len(ml_df))
print("Columns:", len(ml_df.columns))

ML dataframe created.
Rows: 50
Columns: 40


In [106]:
# ---------------------------------------------------------
# One-hot encode the weather categorical features.
#
# Each category gets its own binary column.
# This avoids creating an artificial numerical ordering.
# ---------------------------------------------------------

ml_df = pd.get_dummies(
    ml_df,
    columns=[
        "weather_condition",
        "weather_description"
    ],
    dtype=int
)

print("Categorical encoding completed.")
print("Rows:", len(ml_df))
print("Columns:", len(ml_df.columns))

print("\nNew weather columns:")

weather_encoded_columns = [
    column
    for column in ml_df.columns
    if column.startswith("weather_condition_")
    or column.startswith("weather_description_")
]

print(weather_encoded_columns)

Categorical encoding completed.
Rows: 50
Columns: 48

New weather columns:
['weather_condition_Clear', 'weather_condition_Clouds', 'weather_condition_Rain', 'weather_description_broken clouds', 'weather_description_clear sky', 'weather_description_few clouds', 'weather_description_light rain', 'weather_description_moderate rain', 'weather_description_overcast clouds', 'weather_description_scattered clouds']


In [107]:
# ---------------------------------------------------------
# Separate the destination name from the ML features.
#
# We keep destination separately because it identifies the
# item being recommended. The ML algorithm should work with
# the actual numerical/categorical feature representation.
# ---------------------------------------------------------

# Keep destination names for later recommendation results.
destination_names = ml_df["destination"].copy()

# Remove destination from the feature matrix.
X = ml_df.drop(columns=["destination"])

print("Feature matrix created.")
print("Rows:", X.shape[0])
print("Features:", X.shape[1])

print("\nFeature columns:")
print(X.columns.tolist())

Feature matrix created.
Rows: 50
Features: 47

Feature columns:
['sight_count', 'park_count', 'restaurant_count', 'water_count', 'forest_count', 'wetland_count', 'river_count', 'mountain_count', 'coastal_count', 'sand_count', 'protected_area_count', 'flight_count', 'min_flight_price', 'avg_flight_price', 'max_flight_price', 'avg_total_duration', 'avg_outbound_stops', 'avg_return_stops', 'weather_available', 'accommodation_available', 'flight_available', 'latitude', 'longitude', 'temperature', 'feels_like', 'humidity', 'pressure', 'wind_speed', 'cloudiness', 'visibility', 'rain_1h', 'hotel_count', 'room_count', 'min_hotel_price', 'avg_hotel_price', 'max_hotel_price', 'avg_allotment', 'weather_condition_Clear', 'weather_condition_Clouds', 'weather_condition_Rain', 'weather_description_broken clouds', 'weather_description_clear sky', 'weather_description_few clouds', 'weather_description_light rain', 'weather_description_moderate rain', 'weather_description_overcast clouds', 'weather_desc

In [108]:
# ---------------------------------------------------------
# Feature audit
#
# We separate binary features from continuous/numerical
# features before applying scaling.
# ---------------------------------------------------------

# Binary availability features.
binary_columns = [
    "weather_available",
    "accommodation_available",
    "flight_available"
]

# One-hot encoded weather features are also binary.
weather_binary_columns = [
    column
    for column in X.columns
    if column.startswith("weather_condition_")
    or column.startswith("weather_description_")
]

binary_columns = binary_columns + weather_binary_columns

# Everything else in X is currently numerical/continuous.
numerical_columns = [
    column
    for column in X.columns
    if column not in binary_columns
]

print("Number of binary features:", len(binary_columns))
print("Number of numerical features:", len(numerical_columns))

print("\nBinary features:")
print(binary_columns)

print("\nNumerical features:")
print(numerical_columns)

Number of binary features: 13
Number of numerical features: 34

Binary features:
['weather_available', 'accommodation_available', 'flight_available', 'weather_condition_Clear', 'weather_condition_Clouds', 'weather_condition_Rain', 'weather_description_broken clouds', 'weather_description_clear sky', 'weather_description_few clouds', 'weather_description_light rain', 'weather_description_moderate rain', 'weather_description_overcast clouds', 'weather_description_scattered clouds']

Numerical features:
['sight_count', 'park_count', 'restaurant_count', 'water_count', 'forest_count', 'wetland_count', 'river_count', 'mountain_count', 'coastal_count', 'sand_count', 'protected_area_count', 'flight_count', 'min_flight_price', 'avg_flight_price', 'max_flight_price', 'avg_total_duration', 'avg_outbound_stops', 'avg_return_stops', 'latitude', 'longitude', 'temperature', 'feels_like', 'humidity', 'pressure', 'wind_speed', 'cloudiness', 'visibility', 'rain_1h', 'hotel_count', 'room_count', 'min_hot

In [109]:
# ---------------------------------------------------------
# Scale only the numerical features.
#
# StandardScaler transforms each numerical feature so that
# it has approximately:
#   mean = 0
#   standard deviation = 1
#
# Binary features (0/1) are intentionally NOT scaled.
# ---------------------------------------------------------

from sklearn.preprocessing import StandardScaler

# Create the scaler.
scaler = StandardScaler()

# Create a copy so that we don't accidentally modify X
# while preparing the scaled representation.
X_scaled = X.copy()

# Fit the scaler using only the numerical columns and
# transform those columns.
X_scaled[numerical_columns] = scaler.fit_transform(
    X[numerical_columns]
)

print("Numerical features scaled successfully.")

print("\nShape:")
print(X_scaled.shape)

print("\nFirst 5 rows:")
display(X_scaled.head())

Numerical features scaled successfully.

Shape:
(50, 47)

First 5 rows:


,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,sand_count,...,weather_condition_Clear,weather_condition_Clouds,weather_condition_Rain,weather_description_broken clouds,weather_description_clear sky,weather_description_few clouds,weather_description_light rain,weather_description_moderate rain,weather_description_overcast clouds,weather_description_scattered clouds
0,-0.170163,1.371696,1.169148,0.356656,0.412386,-0.188776,-0.619834,-0.578765,-0.348953,-0.216059,...,0,1,0,0,0,0,0,0,1,0
1,-0.207479,1.042357,-0.427147,-0.601729,-0.795777,-0.493254,-0.406098,-0.578765,-0.348953,-0.216059,...,0,1,0,0,0,0,0,0,1,0
2,-0.020897,-0.645504,0.398523,1.451954,-0.795777,1.841075,1.090052,-0.578765,0.022274,-0.216059,...,0,0,1,0,0,0,1,0,0,0
3,-0.282112,-0.522002,0.013211,0.322428,-0.513872,-0.391761,-0.833569,-0.578765,-0.348953,-0.216059,...,0,1,0,0,0,1,0,0,0,0
4,-0.655276,-1.098345,-1.528040,-1.080922,-0.795777,-0.493254,-0.833569,-0.578765,-0.348953,-0.216059,...,0,0,1,0,0,0,1,0,0,0


In [110]:
# ---------------------------------------------------------
# Verify that numerical features were actually standardized.
# Their means should be very close to 0 and standard
# deviations should be very close to 1.
# ---------------------------------------------------------

print("Mean of scaled numerical features:")
display(
    X_scaled[numerical_columns]
    .mean()
    .round(6)
)

print("\nStandard deviation of scaled numerical features:")
display(
    X_scaled[numerical_columns]
    .std()
    .round(6)
)

Mean of scaled numerical features:


sight_count             0.0
park_count              0.0
restaurant_count       -0.0
water_count             0.0
forest_count           -0.0
wetland_count           0.0
river_count             0.0
mountain_count         -0.0
coastal_count           0.0
sand_count              0.0
protected_area_count    0.0
flight_count            0.0
min_flight_price       -0.0
avg_flight_price       -0.0
max_flight_price       -0.0
avg_total_duration      0.0
avg_outbound_stops      0.0
avg_return_stops       -0.0
latitude                0.0
longitude              -0.0
temperature             0.0
feels_like             -0.0
humidity               -0.0
pressure                0.0
wind_speed             -0.0
cloudiness             -0.0
visibility             -0.0
rain_1h                 0.0
hotel_count            -0.0
room_count              0.0
min_hotel_price         0.0
avg_hotel_price         0.0
max_hotel_price         0.0
avg_allotment          -0.0
dtype: float64


Standard deviation of scaled numerical features:


sight_count             1.010153
park_count              1.010153
restaurant_count        1.010153
water_count             1.010153
forest_count            1.010153
wetland_count           1.010153
river_count             1.010153
mountain_count          1.010153
coastal_count           1.010153
sand_count              1.010153
protected_area_count    1.010153
flight_count            1.010153
min_flight_price        1.010153
avg_flight_price        1.010153
max_flight_price        1.010153
avg_total_duration      1.010153
avg_outbound_stops      1.010153
avg_return_stops        1.010153
latitude                1.010153
longitude               1.010153
temperature             1.010153
feels_like              1.010153
humidity                1.010153
pressure                1.010153
wind_speed              1.010153
cloudiness              1.010153
visibility              1.010153
rain_1h                 1.010153
hotel_count             1.010153
room_count              1.010153
min_hotel_

In [111]:
# ---------------------------------------------------------
# Calculate correlations between numerical features.
#
# We use the original numerical values here rather than
# the scaled values because StandardScaler does not change
# Pearson correlations.
# ---------------------------------------------------------

correlation_matrix = X[numerical_columns].corr()

print("Correlation matrix shape:")
print(correlation_matrix.shape)

display(correlation_matrix.head())

Correlation matrix shape:
(34, 34)


,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,sand_count,...,wind_speed,cloudiness,visibility,rain_1h,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
sight_count,1.000000,0.330961,0.462831,0.340692,-0.072324,0.151563,0.484740,0.101837,0.060587,-0.047561,...,0.363471,0.001798,0.092755,-0.114835,0.616768,0.653111,-0.071787,0.028971,0.015570,0.542938
park_count,0.330961,1.000000,0.570257,0.449081,-0.182363,-0.003363,0.275477,-0.095331,0.064504,0.017771,...,0.072956,0.081415,0.135673,0.049253,0.395404,0.540786,-0.110545,0.053571,0.090714,0.341030
restaurant_count,0.462831,0.570257,1.000000,0.474445,0.047754,0.021433,0.480671,0.144723,0.302743,0.018879,...,0.259267,-0.037339,0.141481,-0.181869,0.432001,0.448567,0.155320,0.296739,0.252853,0.466938
water_count,0.340692,0.449081,0.474445,1.000000,-0.263393,0.064549,0.241113,-0.204319,0.233731,-0.026475,...,0.042230,0.173769,0.089833,0.031876,0.285112,0.275682,-0.020426,0.029952,0.004331,0.135343
forest_count,-0.072324,-0.182363,0.047754,-0.263393,1.000000,-0.011173,-0.016561,0.251719,0.070349,0.101193,...,-0.207634,0.128608,-0.076693,-0.159424,0.000680,-0.039615,0.057942,0.001739,-0.072391,-0.016554


In [112]:
# ---------------------------------------------------------
# Find highly correlated numerical feature pairs.
#
# We remove the diagonal and duplicate pairs so that
# each feature pair appears only once.
# ---------------------------------------------------------

import numpy as np

# Get the absolute correlation values.
abs_corr = correlation_matrix.abs()

# Keep only the upper triangle of the matrix.
upper_triangle = abs_corr.where(
    np.triu(
        np.ones(abs_corr.shape),
        k=1
    ).astype(bool)
)

# Convert the matrix into a list of feature pairs.
high_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlations.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

# Sort from strongest to weakest.
high_correlations = high_correlations.sort_values(
    "correlation",
    ascending=False
)

print("Top 20 strongest feature correlations:")

display(
    high_correlations.head(20)
)

Top 20 strongest feature correlations:


,feature_1,feature_2,correlation
456,avg_flight_price,max_flight_price,1.000000
421,min_flight_price,avg_flight_price,1.000000
422,min_flight_price,max_flight_price,1.000000
561,avg_outbound_stops,avg_return_stops,0.961436
701,temperature,feels_like,0.948356
491,max_flight_price,avg_total_duration,0.941830
423,min_flight_price,avg_total_duration,0.941830
457,avg_flight_price,avg_total_duration,0.941830
981,hotel_count,room_count,0.937677
527,avg_total_duration,avg_return_stops,0.922857


In [113]:
# ---------------------------------------------------------
# Remove redundant flight-price features.
#
# min_flight_price, avg_flight_price and max_flight_price
# are perfectly correlated in our collected dataset.
#
# We keep avg_flight_price as the single representative
# flight-price feature.
# ---------------------------------------------------------

redundant_columns = [
    "min_flight_price",
    "max_flight_price"
]

X_reduced = X_scaled.drop(
    columns=redundant_columns
)

print("Redundant features removed.")

print("Original feature count:", X_scaled.shape[1])
print("Reduced feature count:", X_reduced.shape[1])

print("\nRemoved features:")
print(redundant_columns)

Redundant features removed.
Original feature count: 47
Reduced feature count: 45

Removed features:
['min_flight_price', 'max_flight_price']


In [117]:
# ---------------------------------------------------------
# VERIFY CURRENT FEATURE COLUMNS
#
# X_reduced should contain the 45 features remaining after
# removing only min_flight_price and max_flight_price.
# ---------------------------------------------------------

print("Shape of X_reduced:", X_reduced.shape)

print("\nNumber of features:", len(X_reduced.columns))

print("\nFeature columns:")
for i, column in enumerate(X_reduced.columns, start=1):
    print(f"{i:2}. {column}")

# ---------------------------------------------------------
# Check for missing values
# ---------------------------------------------------------

print("\nMissing values:", X_reduced.isna().sum().sum())

# ---------------------------------------------------------
# Check for duplicate columns
# ---------------------------------------------------------

print("Duplicate columns:", X_reduced.columns.duplicated().sum())

# ---------------------------------------------------------
# Confirm the flight-price columns
# ---------------------------------------------------------

print("\nFlight price columns currently present:")

flight_price_columns = [
    column for column in X_reduced.columns
    if "flight_price" in column
]

print(flight_price_columns)

Shape of X_reduced: (50, 45)

Number of features: 45

Feature columns:
 1. sight_count
 2. park_count
 3. restaurant_count
 4. water_count
 5. forest_count
 6. wetland_count
 7. river_count
 8. mountain_count
 9. coastal_count
10. sand_count
11. protected_area_count
12. flight_count
13. avg_flight_price
14. avg_total_duration
15. avg_outbound_stops
16. avg_return_stops
17. weather_available
18. accommodation_available
19. flight_available
20. latitude
21. longitude
22. temperature
23. feels_like
24. humidity
25. pressure
26. wind_speed
27. cloudiness
28. visibility
29. rain_1h
30. hotel_count
31. room_count
32. min_hotel_price
33. avg_hotel_price
34. max_hotel_price
35. avg_allotment
36. weather_condition_Clear
37. weather_condition_Clouds
38. weather_condition_Rain
39. weather_description_broken clouds
40. weather_description_clear sky
41. weather_description_few clouds
42. weather_description_light rain
43. weather_description_moderate rain
44. weather_description_overcast clouds
45.

In [118]:
# ---------------------------------------------------------
# PCA VARIANCE ANALYSIS
#
# X_reduced contains our verified 45 features.
# We first fit PCA with all components so we can determine
# how many components are required to retain most of the
# information in the dataset.
# ---------------------------------------------------------

from sklearn.decomposition import PCA
import numpy as np

# Fit PCA using all 45 current features.
pca_full = PCA()

X_pca_full = pca_full.fit_transform(X_reduced)

# Calculate cumulative explained variance.
cumulative_variance = np.cumsum(
    pca_full.explained_variance_ratio_
)

# Find the minimum number of components that retain
# at least 90% of the variance.
n_components_90 = np.argmax(
    cumulative_variance >= 0.90
) + 1

print("Original number of features:", X_reduced.shape[1])
print("Components needed for 90% variance:", n_components_90)

print("\nExplained variance by component:")

for i, variance in enumerate(
    pca_full.explained_variance_ratio_,
    start=1
):
    print(
        f"Component {i:2}: "
        f"{variance:.4f} "
        f"(cumulative: {cumulative_variance[i-1]:.4f})"
    )

Original number of features: 45
Components needed for 90% variance: 15

Explained variance by component:
Component  1: 0.1805 (cumulative: 0.1805)
Component  2: 0.1129 (cumulative: 0.2934)
Component  3: 0.0909 (cumulative: 0.3843)
Component  4: 0.0853 (cumulative: 0.4696)
Component  5: 0.0704 (cumulative: 0.5400)
Component  6: 0.0596 (cumulative: 0.5996)
Component  7: 0.0546 (cumulative: 0.6542)
Component  8: 0.0462 (cumulative: 0.7003)
Component  9: 0.0425 (cumulative: 0.7429)
Component 10: 0.0349 (cumulative: 0.7778)
Component 11: 0.0328 (cumulative: 0.8105)
Component 12: 0.0276 (cumulative: 0.8382)
Component 13: 0.0255 (cumulative: 0.8637)
Component 14: 0.0220 (cumulative: 0.8857)
Component 15: 0.0198 (cumulative: 0.9055)
Component 16: 0.0139 (cumulative: 0.9194)
Component 17: 0.0129 (cumulative: 0.9323)
Component 18: 0.0124 (cumulative: 0.9447)
Component 19: 0.0098 (cumulative: 0.9545)
Component 20: 0.0076 (cumulative: 0.9620)
Component 21: 0.0073 (cumulative: 0.9693)
Component 22:

In [119]:
# ---------------------------------------------------------
# CREATE FINAL PCA FEATURE MATRIX
#
# We selected 15 components because they retain 90.55%
# of the variance in our original 45 features.
#
# These 15 components will be used as the compact feature
# representation for our destination recommendation model.
# ---------------------------------------------------------

from sklearn.decomposition import PCA
import pandas as pd

# ---------------------------------------------------------
# Create PCA with the selected number of components.
# ---------------------------------------------------------

pca = PCA(n_components=15)

# Transform our reduced feature matrix.
X_pca = pca.fit_transform(X_reduced)

# ---------------------------------------------------------
# Give meaningful names to the PCA components.
# ---------------------------------------------------------

pca_columns = [
    f"PC{i}"
    for i in range(1, 16)
]

# Convert the NumPy array into a DataFrame.
X_pca_df = pd.DataFrame(
    X_pca,
    columns=pca_columns
)

# ---------------------------------------------------------
# Display the result.
# ---------------------------------------------------------

print("Final PCA feature matrix created.")

print("Rows:", X_pca_df.shape[0])
print("Components:", X_pca_df.shape[1])

print("\nShape:")
print(X_pca_df.shape)

print("\nFirst 5 rows:")
display(X_pca_df.head())

# ---------------------------------------------------------
# Verify total variance retained.
# ---------------------------------------------------------

explained_variance = pca.explained_variance_ratio_.sum()

print(
    f"\nTotal variance retained: "
    f"{explained_variance:.4f} "
    f"({explained_variance * 100:.2f}%)"
)

Final PCA feature matrix created.
Rows: 50
Components: 15

Shape:
(50, 15)

First 5 rows:


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
0,-0.035769,-0.922579,0.051091,-0.643779,-0.734244,0.530902,1.151835,-1.662077,-0.127891,-1.627351,-0.589135,-0.192561,-0.186098,0.064635,-0.709079
1,0.395989,-1.103311,0.733874,0.233504,-0.321427,0.117735,-0.474759,-1.269103,-0.956262,-0.421330,-0.012701,0.246444,-0.413503,0.398685,-1.803953
2,0.359290,-1.191439,-1.151328,1.623510,0.750657,-1.133169,0.889291,2.513876,-0.034903,-0.136900,0.314258,0.235412,-0.979846,0.870892,-0.344422
3,0.127812,-1.082204,3.381689,-0.749300,-1.121050,-0.268471,0.220997,-0.312311,-0.572289,-0.925498,0.318953,1.007212,-0.333766,0.072594,0.400718
4,-2.883816,-0.471556,-1.032682,0.607836,-1.969158,-0.993851,-0.191215,1.046010,0.353361,1.094386,-0.126005,-0.493596,-1.616061,-2.075172,-0.316181



Total variance retained: 0.9055 (90.55%)


In [120]:
# ---------------------------------------------------------
# ATTACH DESTINATION NAMES TO PCA FEATURES
#
# X_pca_df contains the 15 PCA components, but does not
# contain the destination names.
#
# We retrieve the destination column from the same dataframe
# that was used to create the feature matrix.
# ---------------------------------------------------------

# Create a copy of the destination names.
destination_names = integrated_df["destination"].reset_index(drop=True)

# Make sure the number of destinations matches the PCA rows.
print("Number of destinations:", len(destination_names))
print("Number of PCA rows:", len(X_pca_df))

# Combine destination names with PCA components.
pca_destination_df = pd.concat(
    [
        destination_names,
        X_pca_df.reset_index(drop=True)
    ],
    axis=1
)

print("\nPCA destination dataset created.")

print("Rows:", pca_destination_df.shape[0])
print("Columns:", pca_destination_df.shape[1])

print("\nFirst 10 destinations:")
display(pca_destination_df.head(10))

Number of destinations: 50
Number of PCA rows: 50

PCA destination dataset created.
Rows: 50
Columns: 16

First 10 destinations:


,destination,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
0,Agra,-0.035769,-0.922579,0.051091,-0.643779,-0.734244,0.530902,1.151835,-1.662077,-0.127891,-1.627351,-0.589135,-0.192561,-0.186098,0.064635,-0.709079
1,Ahmedabad,0.395989,-1.103311,0.733874,0.233504,-0.321427,0.117735,-0.474759,-1.269103,-0.956262,-0.421330,-0.012701,0.246444,-0.413503,0.398685,-1.803953
2,Alappuzha,0.359290,-1.191439,-1.151328,1.623510,0.750657,-1.133169,0.889291,2.513876,-0.034903,-0.136900,0.314258,0.235412,-0.979846,0.870892,-0.344422
3,Amritsar,0.127812,-1.082204,3.381689,-0.749300,-1.121050,-0.268471,0.220997,-0.312311,-0.572289,-0.925498,0.318953,1.007212,-0.333766,0.072594,0.400718
4,Andaman,-2.883816,-0.471556,-1.032682,0.607836,-1.969158,-0.993851,-0.191215,1.046010,0.353361,1.094386,-0.126005,-0.493596,-1.616061,-2.075172,-0.316181
5,Bengaluru,4.721464,-0.779776,-2.435892,-0.607145,0.407120,1.082846,-1.394853,-0.137190,0.020042,-0.276227,-0.338090,0.155162,-1.201845,-0.410299,0.834461
6,Bhopal,-2.463708,0.627703,-0.906968,-1.412593,-1.054552,-1.124872,0.255230,-0.029381,-0.694939,2.690167,-0.285416,-0.021003,1.371925,0.742714,-0.315462
7,Bhubaneswar,-1.035775,-1.039482,0.133211,-1.488292,-1.516893,-1.509833,1.695194,-0.217925,-0.343341,1.601399,-2.854217,-1.574782,0.421858,0.476566,-0.780953
8,Chennai,2.485913,-3.208803,-1.348073,0.778006,-0.566702,0.552869,1.429000,-0.254685,0.403891,-1.306759,-0.135998,0.488632,0.262385,-0.143057,0.492040
9,Coorg,-2.908784,0.655300,-1.410773,0.998756,-0.470961,-1.263346,-1.204141,0.533750,-0.434169,0.000344,0.609775,-0.085105,-0.432358,-0.153791,-0.809741


In [121]:
# ---------------------------------------------------------
# FINAL PCA DATASET VALIDATION
#
# Before building the recommendation engine, verify that:
# 1. Every destination is unique.
# 2. There are exactly 50 destinations.
# 3. No PCA values are missing.
# 4. The dataset has the expected 16 columns.
# ---------------------------------------------------------

print("Rows:", len(pca_destination_df))
print("Columns:", len(pca_destination_df.columns))

print("\nUnique destinations:",
      pca_destination_df["destination"].nunique())

print("Duplicate destinations:",
      pca_destination_df["destination"].duplicated().sum())

print("\nMissing values:")
print(pca_destination_df.isna().sum())

print("\nPCA columns:")
print(pca_columns)

# ---------------------------------------------------------
# Check that all expected PCA columns exist.
# ---------------------------------------------------------

missing_pca_columns = [
    column
    for column in pca_columns
    if column not in pca_destination_df.columns
]

print("\nMissing PCA columns:", missing_pca_columns)

# ---------------------------------------------------------
# Final validation message
# ---------------------------------------------------------

if (
    len(pca_destination_df) == 50
    and pca_destination_df["destination"].nunique() == 50
    and pca_destination_df["destination"].duplicated().sum() == 0
    and pca_destination_df[pca_columns].isna().sum().sum() == 0
    and len(missing_pca_columns) == 0
):
    print("\n✓ PCA dataset validation PASSED")
else:
    print("\n✗ PCA dataset validation FAILED")

Rows: 50
Columns: 16

Unique destinations: 50
Duplicate destinations: 0

Missing values:
destination    0
PC1            0
PC2            0
PC3            0
PC4            0
PC5            0
PC6            0
PC7            0
PC8            0
PC9            0
PC10           0
PC11           0
PC12           0
PC13           0
PC14           0
PC15           0
dtype: int64

PCA columns:
['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15']

Missing PCA columns: []

✓ PCA dataset validation PASSED


In [122]:
# ---------------------------------------------------------
# FINAL DATA-PROCESSING CHECK
#
# Before saving anything, verify that the expected
# model-ready variables exist in this notebook.
# ---------------------------------------------------------

print("pca_destination_df exists:", "pca_destination_df" in globals())
print("X_scaled exists:", "X_scaled" in globals())
print("pca_columns exists:", "pca_columns" in globals())
print("scaler exists:", "scaler" in globals())
print("pca exists:", "pca" in globals())

pca_destination_df exists: True
X_scaled exists: True
pca_columns exists: True
scaler exists: True
pca exists: True


In [123]:
# ---------------------------------------------------------
# FINAL MODEL-READY DATA SAVE
#
# This cell saves:
# 1. The 45-feature scaled dataset before PCA
# 2. The final 15-component PCA dataset
# 3. The fitted scaler
# 4. The fitted PCA model
# 5. The exact feature order used before PCA
# ---------------------------------------------------------

import os
import joblib
import pandas as pd

# ---------------------------------------------------------
# 1. Create required project directories
# ---------------------------------------------------------

os.makedirs("../data/cleaned", exist_ok=True)
os.makedirs("../models", exist_ok=True)


# ---------------------------------------------------------
# 2. Save the scaled feature matrix
#
# X_scaled contains the 45 features after:
# - categorical encoding
# - numerical scaling
# - before PCA
#
# We add destination names so we can identify each row.
# ---------------------------------------------------------

model_features_scaled = X_scaled.copy()

model_features_scaled.insert(
    0,
    "destination",
    pca_destination_df["destination"].values
)

model_features_scaled.to_csv(
    "../data/cleaned/model_features_scaled.csv",
    index=False
)


# ---------------------------------------------------------
# 3. Save the final PCA dataset
#
# This is the main dataset that the recommendation
# model will use.
# ---------------------------------------------------------

pca_destination_df.to_csv(
    "../data/cleaned/pca_destination_features.csv",
    index=False
)


# ---------------------------------------------------------
# 4. Save the fitted StandardScaler
#
# We will use this SAME scaler later when converting
# a new user's input into the model's feature space.
# ---------------------------------------------------------

joblib.dump(
    scaler,
    "../models/feature_scaler.pkl"
)


# ---------------------------------------------------------
# 5. Save the fitted PCA model
#
# This ensures that future user inputs are transformed
# using exactly the same PCA transformation.
# ---------------------------------------------------------

joblib.dump(
    pca,
    "../models/pca_model.pkl"
)


# ---------------------------------------------------------
# 6. Save the original feature order
#
# This is extremely important.
# New user data must have exactly the same feature order
# before scaling and PCA transformation.
# ---------------------------------------------------------

joblib.dump(
    pca_columns,
    "../models/feature_columns.pkl"
)


# ---------------------------------------------------------
# 7. Final confirmation
# ---------------------------------------------------------

print("=" * 60)
print("MODEL ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 60)

print("\nScaled feature dataset:")
print("../data/cleaned/model_features_scaled.csv")

print("\nPCA dataset:")
print("../data/cleaned/pca_destination_features.csv")

print("\nScaler:")
print("../models/feature_scaler.pkl")

print("\nPCA model:")
print("../models/pca_model.pkl")

print("\nFeature order:")
print("../models/feature_columns.pkl")

print("\nPCA dataset shape:", pca_destination_df.shape)
print("Scaled dataset shape:", model_features_scaled.shape)

MODEL ARTIFACTS SAVED SUCCESSFULLY

Scaled feature dataset:
../data/cleaned/model_features_scaled.csv

PCA dataset:
../data/cleaned/pca_destination_features.csv

Scaler:
../models/feature_scaler.pkl

PCA model:
../models/pca_model.pkl

Feature order:
../models/feature_columns.pkl

PCA dataset shape: (50, 16)
Scaled dataset shape: (50, 48)


In [124]:
# ---------------------------------------------------------
# Verify that all model-ready files were created
# ---------------------------------------------------------

import os

files_to_check = [
    "../data/cleaned/model_features_scaled.csv",
    "../data/cleaned/pca_destination_features.csv",
    "../models/feature_scaler.pkl",
    "../models/pca_model.pkl",
    "../models/feature_columns.pkl"
]

for file_path in files_to_check:
    print(
        "✓" if os.path.exists(file_path) else "✗",
        file_path
    )

✓ ../data/cleaned/model_features_scaled.csv
✓ ../data/cleaned/pca_destination_features.csv
✓ ../models/feature_scaler.pkl
✓ ../models/pca_model.pkl
✓ ../models/feature_columns.pkl


In [ ]:
# ---------------------------------------------------------
# RECOVERY CELL
#
# The Jupyter kernel was restarted, so all Python variables
# were cleared from memory.
#
# We reload the already-saved model artifacts from disk.
# This does NOT recollect data from APIs.
# ---------------------------------------------------------

import pandas as pd
import joblib


# ---------------------------------------------------------
# Load the saved datasets.
# ---------------------------------------------------------

model_features_scaled_df = pd.read_csv(
    "../data/cleaned/model_features_scaled.csv"
)

pca_destination_df = pd.read_csv(
    "../data/cleaned/pca_destination_features.csv"
)


# ---------------------------------------------------------
# Load the preprocessing artifacts.
# ---------------------------------------------------------

scaler = joblib.load(
    "../models/feature_scaler.pkl"
)

pca = joblib.load(
    "../models/pca_model.pkl"
)

feature_columns = joblib.load(
    "../models/feature_columns.pkl"
)

numerical_feature_columns = joblib.load(
    "../models/numerical_feature_columns.pkl"
)

binary_feature_columns = joblib.load(
    "../models/binary_feature_columns.pkl"
)

redundant_columns = joblib.load(
    "../models/redundant_features.pkl"
)


# ---------------------------------------------------------
# Display what was recovered.
# ---------------------------------------------------------

print("Saved processing artifacts reloaded successfully.")

print("\nScaled dataset:")
print("Rows:", model_features_scaled_df.shape[0])
print("Columns:", model_features_scaled_df.shape[1])

print("\nPCA destination dataset:")
print("Rows:", pca_destination_df.shape[0])
print("Columns:", pca_destination_df.shape[1])

print("\nOriginal features:", len(feature_columns))
print("Numerical features:", len(numerical_feature_columns))
print("Binary features:", len(binary_feature_columns))
print("Redundant features:", len(redundant_columns))

print("\nScaler expects:",
      scaler.n_features_in_,
      "features")

print("PCA expects:",
      pca.n_features_in_,
      "features")

print("PCA components:",
      pca.n_components_)

Saved processing artifacts reloaded successfully.

Scaled dataset:
Rows: 50
Columns: 48

PCA destination dataset:
Rows: 50
Columns: 16

Original features: 47
Numerical features: 34
Binary features: 13
Redundant features: 2

Scaler expects: 34 features
PCA expects: 45 features
PCA components: 15


In [3]:
# ---------------------------------------------------------
# RECOVERY CELL
#
# The Jupyter kernel was restarted, so all Python variables
# were cleared from memory.
#
# We reload the already-saved model artifacts from disk.
# This does NOT recollect data from APIs.
# ---------------------------------------------------------

import pandas as pd
import joblib


# ---------------------------------------------------------
# Load the saved datasets.
# ---------------------------------------------------------

model_features_scaled_df = pd.read_csv(
    "../data/cleaned/model_features_scaled.csv"
)

pca_destination_df = pd.read_csv(
    "../data/cleaned/pca_destination_features.csv"
)


# ---------------------------------------------------------
# Load the preprocessing artifacts.
# ---------------------------------------------------------

scaler = joblib.load(
    "../models/feature_scaler.pkl"
)

pca = joblib.load(
    "../models/pca_model.pkl"
)

feature_columns = joblib.load(
    "../models/feature_columns.pkl"
)

numerical_feature_columns = joblib.load(
    "../models/numerical_feature_columns.pkl"
)

binary_feature_columns = joblib.load(
    "../models/binary_feature_columns.pkl"
)

redundant_columns = joblib.load(
    "../models/redundant_features.pkl"
)


# ---------------------------------------------------------
# Display what was recovered.
# ---------------------------------------------------------

print("Saved processing artifacts reloaded successfully.")

print("\nScaled dataset:")
print("Rows:", model_features_scaled_df.shape[0])
print("Columns:", model_features_scaled_df.shape[1])

print("\nPCA destination dataset:")
print("Rows:", pca_destination_df.shape[0])
print("Columns:", pca_destination_df.shape[1])

print("\nOriginal features:", len(feature_columns))
print("Numerical features:", len(numerical_feature_columns))
print("Binary features:", len(binary_feature_columns))
print("Redundant features:", len(redundant_columns))

print("\nScaler expects:",
      scaler.n_features_in_,
      "features")

print("PCA expects:",
      pca.n_features_in_,
      "features")

print("PCA components:",
      pca.n_components_)

Saved processing artifacts reloaded successfully.

Scaled dataset:
Rows: 50
Columns: 48

PCA destination dataset:
Rows: 50
Columns: 16

Original features: 47
Numerical features: 34
Binary features: 13
Redundant features: 2

Scaler expects: 34 features
PCA expects: 45 features
PCA components: 15


In [4]:
# ---------------------------------------------------------
# FINAL CHECKPOINT - DATA PROCESSING COMPLETE
#
# This cell confirms that all saved artifacts required by
# the model-building stage exist and are internally
# consistent.
# ---------------------------------------------------------

import os

required_files = {
    "Scaled feature data":
        "../data/cleaned/model_features_scaled.csv",

    "PCA destination data":
        "../data/cleaned/pca_destination_features.csv",

    "Feature scaler":
        "../models/feature_scaler.pkl",

    "PCA model":
        "../models/pca_model.pkl",

    "Original feature columns":
        "../models/feature_columns.pkl",

    "Numerical feature columns":
        "../models/numerical_feature_columns.pkl",

    "Binary feature columns":
        "../models/binary_feature_columns.pkl",

    "Redundant feature list":
        "../models/redundant_features.pkl"
}


print("=" * 60)
print("FINAL DATA PROCESSING CHECKPOINT")
print("=" * 60)

all_files_exist = True

for name, path in required_files.items():

    if os.path.exists(path):
        print(f"✓ {name}")
    else:
        print(f"✗ {name} MISSING")
        all_files_exist = False


print("\n" + "=" * 60)

if all_files_exist:
    print("✓ DATA PROCESSING PIPELINE COMPLETE")
    print("✓ ALL MODEL ARTIFACTS AVAILABLE")
    print("✓ READY FOR MODEL BUILDING")
else:
    print("✗ SOME ARTIFACTS ARE MISSING")

FINAL DATA PROCESSING CHECKPOINT
✓ Scaled feature data
✓ PCA destination data
✓ Feature scaler
✓ PCA model
✓ Original feature columns
✓ Numerical feature columns
✓ Binary feature columns
✓ Redundant feature list

✓ DATA PROCESSING PIPELINE COMPLETE
✓ ALL MODEL ARTIFACTS AVAILABLE
✓ READY FOR MODEL BUILDING
